In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def add_features(df):
    # the model sees that after december comes january and after 23(h) 0
    df["hr_sin"] = np.sin(2*np.pi*df["hr"]/24)
    df["hr_cos"] = np.cos(2*np.pi*df["hr"]/24)

    df["month_sin"] = np.sin(2*np.pi*df["mnth"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["mnth"]/12)

    df["is_weekend"] = (df["weekday"] >= 5).astype(int)

    return df

In [ ]:
class BikeDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        if y is not None:
            self.y = torch.tensor(y, dtype=torch.float32)
        else:
            self.y = None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

In [ ]:
class BikeModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim,256),
            nn.ReLU(),
            nn.BatchNorm1d(256),

            nn.Linear(256,128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,1)
        )

    def forward(self,x):
        return self.net(x).squeeze()

In [ ]:
data = pd.read_csv("data.csv")
data = add_features(data)

X = data.drop(columns=["casual","registered","cnt","dteday","instant","yr"])
# it's easier for the model to predict log
y = np.log1p(data["cnt"])

In [7]:
X_train,X_val,y_train,y_val = train_test_split(
    X,y,test_size=0.2,random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [8]:
train_dataset = BikeDataset(X_train,y_train.values)
val_dataset = BikeDataset(X_val,y_val.values)

train_loader = DataLoader(train_dataset,batch_size=128,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=128,shuffle=False)

In [9]:
model = BikeModel(X_train.shape[1]).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
loss_fn = nn.MSELoss()

In [10]:
EPOCHS = 80

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for X_batch,y_batch in train_loader:

        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = loss_fn(pred,y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    # val

    model.eval()
    val_loss = 0

    with torch.no_grad():

        for X_batch,y_batch in val_loader:

            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            pred = model(X_batch)

            loss = loss_fn(pred,y_batch)

            val_loss += loss.item()

    print(
        f"Epoch {epoch+1} : "
        f"train {train_loss/len(train_loader):.4f} : "
        f"val {val_loss/len(val_loader):.4f}"
    )


Epoch 1 : train 4.0281 : val 0.5492
Epoch 2 : train 0.4806 : val 0.3545
Epoch 3 : train 0.3349 : val 0.3638
Epoch 4 : train 0.2743 : val 0.2642
Epoch 5 : train 0.2545 : val 0.2517
Epoch 6 : train 0.2522 : val 0.2221
Epoch 7 : train 0.2415 : val 0.2188
Epoch 8 : train 0.2394 : val 0.2153
Epoch 9 : train 0.2522 : val 0.1978
Epoch 10 : train 0.2737 : val 0.2207
Epoch 11 : train 0.3073 : val 0.2945
Epoch 12 : train 0.3015 : val 0.2200
Epoch 13 : train 0.2451 : val 0.1780
Epoch 14 : train 0.1995 : val 0.1906
Epoch 15 : train 0.1806 : val 0.1811
Epoch 16 : train 0.1672 : val 0.1745
Epoch 17 : train 0.2032 : val 0.1895
Epoch 18 : train 0.1796 : val 0.2314
Epoch 19 : train 0.1734 : val 0.1895
Epoch 20 : train 0.2091 : val 0.1895
Epoch 21 : train 0.1757 : val 0.1837
Epoch 22 : train 0.1808 : val 0.2226
Epoch 23 : train 0.1912 : val 0.1972
Epoch 24 : train 0.1436 : val 0.1763
Epoch 25 : train 0.1378 : val 0.1642
Epoch 26 : train 0.1473 : val 0.1871
Epoch 27 : train 0.1394 : val 0.2158
Epoch 28 :

In [11]:
eval_data = pd.read_csv("evaluation_data.csv")
eval_data = add_features(eval_data)

X_test = eval_data.drop(columns=["dteday","yr"])
X_test = scaler.transform(X_test)

test_dataset = BikeDataset(X_test)
test_loader = DataLoader(test_dataset,batch_size=128)

In [ ]:
model.eval()
predictions = []

with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        pred = model(X_batch)
        # exp because we used log earlier
        pred = torch.expm1(pred)
        # max(0,pred)
        pred = torch.clamp(pred,min=0)
        predictions.extend(pred.cpu().numpy())

predictions = np.array(predictions)

In [13]:
np.savetxt(
    "sroda_Nowak_Raczyńska_predictions.csv",
    predictions,
    delimiter=","
)